# Phase 2 — Optimisation ciblée (score + qualité)

Référence phase 1 (`02_optimization_sweep.ipynb`, run Colab 2026-09-12) :
sélection **`c32-lower-alph1000-bf-b3` — score 1.7577, jaccard 0.9652**,
guardrail PASS, checker READY. Meilleur brut : `c30-lower-alph500-bf` — 1.7576.

Ce notebook ne refait PAS le balayage : il rejoue 2 **calibrateurs** (c30/c32,
reproductibilité ±0.002) puis teste 15 configs ciblées (c41–c55) autour de la
recette gagnante : case manquante lower×500×3, boost amharique, alphabets nettoyés,
interactions lowercase, cartographie fine de l'alphabet.

Règles de décision (§16) : REMPLACER seulement si gain ≥ 0.005 ET jaccard tenu
ET UNK = 0 ET marges guardrail ≥ 0.04 ET checker READY. Sinon KEEP (c32).
`submissions/` n'est JAMAIS modifié par ce notebook (promotion manuelle).

## Règles officielles (rappel, vérifiées dans le code du challenge)

| Élément | Valeur officielle |
|---|---|
| Mot | `len(text.split())` (espaces blancs) |
| Fertility | `tokens / words` |
| Pénalité UNK | `100 × (unk_tokens / words)` |
| Score par langue | `fertility + 100 × unk_rate` |
| Score final | moyenne de **ha, sw, yo, am** |
| Guardrail EN/FR | `fertility ≤ 1.15 × moyenne brute des 4 notées` |
| Vocab max / taille max | 10 000 tokens / 20 MiB |
| Version imposée | `tokenizers==0.22.1` |


## 1. Installation (versions officielles)

`tokenizers==0.22.1` est **imposé** par le challenge : le checker officiel vérifie l'égalité exacte
de version (`SUPPORTED_TOKENIZERS_VERSION = "0.22.1"`). On épingle donc cette version.

In [ ]:
!pip install -q "tokenizers==0.22.1" datasets pandas numpy sentencepiece
import tokenizers
print("tokenizers:", tokenizers.__version__, "(attendu 0.22.1)")
assert tokenizers.__version__ == "0.22.1", "Installer tokenizers==0.22.1 (exigence officielle)"

## 2. Constantes, métrique officielle et guardrail

Le code ci-dessous reproduit **exactement** la métrique officielle
(`competition/metrics.py`) et les constantes (`competition/constants.py`).
La cellule définit aussi la **métrique qualité** (Jaccard sur 4 variantes préservant le sens : casse, NFD, espaces multiples, ponctuation détachée) : à score proche (±0.01), la sélection retient le tokenizer le plus robuste.


In [ ]:
# =============================================================================
# 2. Constantes + métrique OFFICIELLES (compétition)
# =============================================================================
import json, os, re, shutil, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

# ---- Constantes officielles (competition/constants.py) ---------------------
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
UNKNOWN_PENALTY = 100.0
MAX_VOCAB_SIZE = 10_000
MAX_TOKENIZER_BYTES = 20 * 1024 * 1024
REQUIRED_TOKENIZERS_VERSION = "0.22.1"
SMOKE_TEXTS = {
    "en": "Knowledge grows when it is shared.",
    "fr": "Le savoir grandit lorsqu’il est partagé.",
    "ha": "Ilimi yana ƙaruwa idan an raba shi.",
    "sw": "Maarifa hukua yanaposhirikishwa.",
    "yo": "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "am": "እውቀት ሲካፈል ያድጋል።",
}

# ---- Dataset officiel ------------------------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# ---- Paramètres du balayage (modifiables) ----------------------------------
VOCAB_SIZE = 10_000            # imposé par le challenge
MAX_TRAIN_DOCS = None          # None = tout le train (240 000) ; ex. 60_000 pour un pré-balayage rapide
BASELINE_REFERENCE_SCORE = 2.059977   # score officiel obtenu par 01_baseline_bpe_10k

OUTPUT_ROOT = Path.cwd()
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSIONS_DIR = OUTPUT_ROOT / "submissions"
for d in (REPORT_DIR, MODEL_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ---- MÉTRIQUE OFFICIELLE ---------------------------------------------------
def count_words(text: str) -> int:
    """Mots = séparés par des espaces blancs (comme l'évaluateur officiel)."""
    return len(text.split())


def unknown_token_id(tokenizer: Tokenizer):
    """Id émis pour un texte non représentable (comme l'évaluateur officiel)."""
    model = json.loads(tokenizer.to_str()).get("model", {})
    name = model.get("unk_token")
    if isinstance(name, str):
        return tokenizer.token_to_id(name)
    unk_id = model.get("unk_id")
    return int(unk_id) if unk_id is not None else None


def measure(rows, tokenizer, batch_size=2048):
    """rows = liste de (language, text) -> fertility, unk_rate, tokens, words, lossy, unk_total."""
    tokens, words, unknowns = defaultdict(int), defaultdict(int), defaultdict(int)
    unknown_id = unknown_token_id(tokenizer)
    lossy = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        encodings = tokenizer.encode_batch([t for _, t in batch], add_special_tokens=False)
        for (lang, text), enc in zip(batch, encodings, strict=True):
            tokens[lang] += len(enc.ids)
            words[lang] += count_words(text)
            if unknown_id is not None:
                unknowns[lang] += sum(1 for v in enc.ids if v == unknown_id)
            if tokenizer.decode(enc.ids, skip_special_tokens=False) != text:
                lossy += 1
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES if words[l]}
    unk_rate = {l: unknowns[l] / words[l] for l in LANGUAGES if words[l]}
    return fertility, unk_rate, dict(tokens), dict(words), lossy, sum(unknowns.values())


def penalised_scores(fertility, unk_rate):
    return {l: v + UNKNOWN_PENALTY * unk_rate.get(l, 0.0) for l, v in fertility.items()}


def competition_score(fertility, unk_rate=None):
    """Score officiel = moyenne des langues notées (ha, sw, yo, am)."""
    missing = [l for l in SCORED_LANGUAGES if l not in fertility]
    if missing:
        raise ValueError(f"langues notées manquantes : {missing}")
    scores = penalised_scores(fertility, unk_rate or {})
    return sum(scores[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)


def guardrail(fertility):
    """budget = 1.15 x moyenne(fertility brute des langues notées)."""
    raw = sum(fertility[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility.get(l, 0.0) > budget]
    return raw, budget, breaches


# ---- MÉTRIQUE QUALITÉ (robustesse aux variantes, Kamali 2026) ----------------
# Le score officiel ne voit pas OÙ tombent les découpes : à score proche (±0.01),
# on sélectionne le tokenizer dont les tokens survivent aux variantes qui
# préservent le sens (casse, NFD, espaces multiples, ponctuation détachée).
QUALITY_SEED = 7
QUALITY_SAMPLES_PER_LANG = 150
QUALITY_TIEBAND = 0.01           # bande autour du meilleur score brut
QUALITY_VARIANTS = ("lower", "nfd", "dblspace", "detachpunct")

# Ponctuation détachée par la variante "detachpunct" (ASCII + guillemets + éthiopienne)
_PUNCT_DETACH = (".,;:!?()[]{}" + "'" + '"'
                 + "\u00ab\u00bb\u2019\u2026\u2013\u2014"
                 + "\u1361\u1362\u1363\u1364\u1365\u1366\u1367\u1368")


def quality_samples(val_rows, per_lang=QUALITY_SAMPLES_PER_LANG, seed=QUALITY_SEED):
    """Échantillon déterministe (graine fixe) pour la métrique qualité."""
    import random
    rng = random.Random(seed)
    by_lang = defaultdict(list)
    for lang, text in val_rows:
        by_lang[lang].append(text)
    samples = []
    for lang in LANGUAGES:
        pool = by_lang[lang]
        for i in rng.sample(range(len(pool)), min(per_lang, len(pool))):
            samples.append((lang, pool[i]))
    return samples


def _detach_punct(text):
    for p in _PUNCT_DETACH:
        if p in text:
            text = text.replace(p, f" {p} ")
    return text


def quality_variants(text):
    """4 variantes préservant le sens (le texte brut sert de référence)."""
    return {
        "lower": text.lower(),
        "nfd": unicodedata.normalize("NFD", text),
        "dblspace": re.sub(r"\s", "  ", text),
        "detachpunct": _detach_punct(text),
    }


def _jaccard(ids_a, ids_b):
    set_a, set_b = set(ids_a), set(ids_b)
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def measure_quality(tokenizer, samples):
    """Jaccard moyen (tokens partagés base/variante) par variante + moyenne."""
    base_ids = [e.ids for e in tokenizer.encode_batch(
        [t for _, t in samples], add_special_tokens=False)]
    per_variant = {}
    for variant in QUALITY_VARIANTS:
        var_ids = [e.ids for e in tokenizer.encode_batch(
            [quality_variants(t)[variant] for _, t in samples], add_special_tokens=False)]
        per_variant[variant] = (sum(_jaccard(a, b) for a, b in zip(base_ids, var_ids))
                                / len(samples))
    per_variant["mean"] = sum(per_variant.values()) / len(per_variant)
    return per_variant


def validate_tokenizer_file(path):
    """Contrôles officiels de validation (competition/validation.py)."""
    path = Path(path)
    checks, errors = {}, []
    checks["file_size"] = path.stat().st_size <= MAX_TOKENIZER_BYTES
    if not checks["file_size"]:
        errors.append("fichier > 20 MiB")
    tok = Tokenizer.from_file(str(path))
    checks["loads"] = True
    vocab_size = tok.get_vocab_size(with_added_tokens=True)
    checks["vocabulary"] = vocab_size <= MAX_VOCAB_SIZE
    if not checks["vocabulary"]:
        errors.append(f"vocabulaire {vocab_size:,} > {MAX_VOCAB_SIZE:,}")
    encodings = tok.encode_batch(list(SMOKE_TEXTS.values()), add_special_tokens=False)
    checks["encodes_all_languages"] = all(e.ids for e in encodings)
    if not checks["encodes_all_languages"]:
        errors.append("une langue ne produit aucun token")
    decoded = [tok.decode(e.ids, skip_special_tokens=False) for e in encodings]
    checks["decodes"] = all(t.strip() for t in decoded)
    if not checks["decodes"]:
        errors.append("un décodage est vide")
    import tokenizers as _tk
    checks["compatible_version"] = _tk.__version__ == REQUIRED_TOKENIZERS_VERSION
    if not checks["compatible_version"]:
        errors.append(f"tokenizers {_tk.__version__} != {REQUIRED_TOKENIZERS_VERSION}")
    lossy_languages = [l for l, orig, rest in zip(SMOKE_TEXTS, SMOKE_TEXTS.values(), decoded)
                       if orig != rest]
    return {"valid": all(checks.values()) and not errors, "checks": checks, "errors": errors,
            "vocab_size": vocab_size, "file_size_bytes": path.stat().st_size,
            "lossy_languages": lossy_languages}


print("Métrique officielle chargée. Vocab max:", MAX_VOCAB_SIZE, "| guardrail ratio:", CONTEXT_FERTILITY_RATIO)
print("Tokenizers:", tokenizers.__version__ if (tokenizers := __import__("tokenizers")) else None)

# ---- RÉFÉRENCE PHASE 1 (c32 publié, run Colab 2026-09-12, tokenizers 0.22.1) --
REF_NAME = "c32-lower-alph1000-bf-b3"
REF_SCORE = 1.7577300319444404
REF_JACCARD = 0.9651731022636671
C30_REF_SCORE = 1.7575868305114262
C30_REF_JACCARD = 0.9641719011986398
CALIBRATORS = ("c30-lower-alph500-bf", "c32-lower-alph1000-bf-b3")
# Seuils de REMPLACEMENT (§16) : gain significatif + qualité + marge + checker.
REPLACE_MIN_GAIN = 0.005        # au-delà du bruit (~0.002)
REPLACE_MIN_MARGIN = 0.04       # marge guardrail minimale (sécurité test caché)
REPLACE_JACC_TOL = 0.005        # tolérance jaccard vs référence
REPRO_TOL = 0.002               # tolérance de reproductibilité des calibrateurs


## 3. Données : chargement + préparation

Entraînement sur `train` uniquement, évaluation sur `validation` uniquement (comme le baseline).

In [ ]:
# =============================================================================
# 3. Chargement du dataset officiel + préparation des textes
# =============================================================================
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(dataset)

train_by_lang = {}
for lang in LANGUAGES:
    train_by_lang[lang] = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
if MAX_TRAIN_DOCS:
    train_by_lang = {l: texts[:MAX_TRAIN_DOCS] for l, texts in train_by_lang.items()}

val_rows = []
for lang in LANGUAGES:
    texts = [t for t, l in zip(dataset["validation"]["text"], dataset["validation"]["language"]) if l == lang]
    val_rows.extend((lang, t) for t in texts)

print("\nEntraînement :", {l: f"{len(v):,}" for l, v in train_by_lang.items()},
      "| total:", f"{sum(len(v) for v in train_by_lang.values()):,}")
print("Validation   :", {l: f"{sum(1 for x, _ in val_rows if x == l):,}" for l in LANGUAGES},
      "| total:", f"{len(val_rows):,}")

quality_rows = quality_samples(val_rows)
print("Échantillon qualité :", f"{len(quality_rows):,}", "textes (150/langue × 6, graine fixe) —",
      len(QUALITY_VARIANTS), "variantes par texte")
assert len(dataset["train"]) == 240_000 or MAX_TRAIN_DOCS, "train inattendu"
assert len(val_rows) == 24_000, "validation inattendue"

## 4. Matrice phase 2 (c41–c55 + 2 calibrateurs)

Toutes les nouvelles configs partent de la recette gagnante (BPE + `nfc_lower` +
WhitespaceSplit + byte fallback) et bougent UN axe à la fois :

- **EXP-200** — `c41` lower×500×3 (LA case manquante), `c42` lower×500×4 (sonde frontière) ;
- **EXP-201** — parité amharique : `c43`–`c45` am ×4/×5/×6, `c46` sw/yo ×3, `c47` ha×1+am×4 ;
- **EXP-202** — alphabets nettoyés : `c48` v1 (arabe gardé), `c49` v2 (sans arabe, sonde),
  `c50` top-500 + tout l'éthiopien ;
- **EXP-203** — interactions : `c51` lower×punctiso, `c52` lower×unigram, `c53` nfkc+lower ;
- **EXP-204** — alphabet fin : `c54` 400×3, `c55` 750×3.

Les calibrateurs c30/c32 valident la reproductibilité avant toute conclusion (§15).


In [ ]:
# =============================================================================
# 4. Définition des configurations candidates
# =============================================================================
from tokenizers.models import BPE, Unigram, WordPiece
from tokenizers.normalizers import NFC, NFKC, Lowercase, Sequence as NormSequence
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPre, Metaspace
from tokenizers import Regex
from tokenizers.pre_tokenizers import Sequence as PreSequence, Split
from tokenizers.pre_tokenizers import Whitespace, WhitespaceSplit
from tokenizers.decoders import ByteFallback, Sequence
from tokenizers.decoders import Metaspace as MetaspaceDec
from tokenizers.decoders import ByteLevel as ByteLevelDec
from tokenizers.trainers import BpeTrainer, UnigramTrainer, WordPieceTrainer

# --- EXP-004 : byte fallback ---------------------------------------------------
# Recette mesurée : les 256 tokens <0xXX> doivent être DANS le vocabulaire du modèle (sinon le
# byte_fallback du modèle ne trouve rien et émet [UNK]) et comptent DANS vocab_size.
# Passer par add_tokens() après entraînement NE fonctionne PAS (mesuré : 41/41 mots -> [UNK]).
BYTE_TOKENS = [f"<0x{i:02X}>" for i in range(256)]


def is_byte_token(token):
    """Vrai pour les tokens byte du type <0xEF> (tokens ordinaires, comme dans Llama-2)."""
    return len(token) == 6 and token.startswith("<0x") and token.endswith(">")


CONFIGS = [
    # Calibrateurs : recettes phase 1 reconstruites depuis zéro (§15 — la rerun
    # doit redonner le score publié à ±0.002, sinon le run n'est pas valide).
    dict(name="c30-lower-alph500-bf", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="CALIBRATEUR : meilleur brut phase 1 (1.7576)"),
    dict(name="c32-lower-alph1000-bf-b3", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=1000, byte_fallback=True,
         boost={"ha": 3, "sw": 3, "yo": 3, "am": 3},
         note="CALIBRATEUR : sélection phase 1 (1.7577, jaccard 0.9652)"),
    # =========================================================================
    # EXP-200 — la case manquante : lowercase × alphabet 500 × boost ×3/×4
    #   c30 (x2) et c32 (x3 mais alph1000) sont à égalité : c41 tranche.
    # =========================================================================
    dict(name="c41-lower-alph500-b3", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 3, "sw": 3, "yo": 3, "am": 3},
         note="LA case manquante : c30 + boost x3 (priorité n°1)"),
    dict(name="c42-lower-alph500-b4", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 4, "sw": 4, "yo": 4, "am": 4},
         note="SONDE frontière : lower+x4 (sélection seulement si marge ≥ 0.04)"),
    # =========================================================================
    # EXP-201 — parité ciblée : l'amharique (2.20) reste 31 % du score.
    #   Le boost am seul est frugal en guardrail (c37 : marge fr +0.145 sans lower).
    # =========================================================================
    dict(name="c43-lower-alph500-am4", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 4},
         note="c30 + amharique x4 seul (bottleneck, frugal en guardrail)"),
    dict(name="c44-lower-alph500-am5", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 5},
         note="c30 + amharique x5 seul"),
    dict(name="c45-lower-alph500-am6", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 6},
         note="SONDE : c30 + amharique x6 seul (rendements décroissants ?)"),
    dict(name="c46-lower-alph500-swyo3", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 3, "yo": 3, "am": 2},
         note="c38 sous lowercase : sw/yo x3 (deuxièmes langues les plus chères)"),
    dict(name="c47-lower-alph500-ha1am4", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 1, "sw": 2, "yo": 2, "am": 4},
         note="réallocation parité : ha déjà à 1.49 → sa part va à am x4"),
    # =========================================================================
    # EXP-202 — alphabets nettoyés : le top-1000 de c32 gaspille ~400 places sur
    #   des résidus Wikipédia (CJK, hangul, grec, cyrillique…) que le byte fallback
    #   couvre déjà. On les rend aux merges utiles (éthiopien, latin, diacritiques).
    # =========================================================================
    dict(name="c48-lower-cleanalpha-v1", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet="clean_v1", byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="alphabet nettoyé v1 (arabe gardé, hypothèse Ajami) + byte fallback"),
    dict(name="c49-lower-cleanalpha-v2", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet="clean_v2", byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="SONDE : alphabet nettoyé v2 SANS arabe (teste l'hypothèse Ajami)"),
    dict(name="c50-lower-geztopup", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet="gez_topup", byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="top-500 + TOUT l'éthiopien du train (couverture guèze totale)"),
    # =========================================================================
    # EXP-203 — interactions non testées : lowercase × frontières / modèle / norme
    # =========================================================================
    dict(name="c51-lower-punctiso-alph500", model="bpe", pre="whitespace_split_punct", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="lower × ponctuation isolée (cartographie score×qualité)"),
    dict(name="c52-lower-uni-alph500", model="unigram", pre="whitespace_split", min_freq=0,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="dernière chance unigram : c33 sous lowercase"),
    dict(name="c53-nfkclower-alph500", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfkc_lower", alphabet=False, limit_alphabet=500, byte_fallback=True,
         boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="NFKC + lowercase (point final normalisation)"),
    # =========================================================================
    # EXP-204 — cartographie fine de l'alphabet sous lowercase ×3
    #   (le genou 250→500→1000 a été mappé sans lower ; on remappe avec lower)
    # =========================================================================
    dict(name="c54-lower-alph400-b3", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=400, byte_fallback=True,
         boost={"ha": 3, "sw": 3, "yo": 3, "am": 3},
         note="alphabet 400 + lower + x3 (entre le genou 250 et l'optimum 500)"),
    dict(name="c55-lower-alph750-b3", model="bpe", pre="whitespace_split", min_freq=5,
         normalizer="nfc_lower", alphabet=False, limit_alphabet=750, byte_fallback=True,
         boost={"ha": 3, "sw": 3, "yo": 3, "am": 3},
         note="alphabet 750 + lower + x3 (entre 500 et 1000)"),
]

# Alfabet : caractères du train (>=1 occurrence) pour 'alphabet=True'
if any(c["alphabet"] is True for c in CONFIGS):
    train_chars = set()
    for texts in train_by_lang.values():
        for t in texts:
            train_chars.update(ch for ch in t if not ch.isspace())
    train_alphabet = sorted(train_chars)
    print(f"Alphabet du train : {len(train_alphabet):,} caractères distincts (hors espaces)")
else:
    train_alphabet = []

# Top-N fréquent pour l'unigram HF (EXP-010) : UnigramTrainer IGNORE limit_alphabet
# (avertissement « Ignored unknown kwargs » mesuré) — on lui fournit explicitement
# les N caractères les plus fréquents comme initial_alphabet.
_uni_limits = sorted({c.get("limit_alphabet") for c in CONFIGS
                      if c.get("model") == "unigram" and c.get("limit_alphabet")})
top_alphabets = {}
if _uni_limits:
    _char_counts = Counter()
    for texts in train_by_lang.values():
        for t in texts:
            _char_counts.update(ch for ch in t if not ch.isspace())
    for n in _uni_limits:
        top_alphabets[n] = [ch for ch, _ in _char_counts.most_common(n)]
    print(f"Top alphabets unigram : { {n: len(a) for n, a in top_alphabets.items()} }")
# Alphabets nettoyés (EXP-202) : on ne garde que les écritures utiles aux 6 langues
# (latin, éthiopien, arabe en v1) + chiffres/ponctuation/symboles/marques courants.
# Tout le reste (CJK, hangul, grec, cyrillique, hébreu, thaï…) reste couvert par le
# byte fallback (2-3 tokens par occurrence rarissime). Déterministe, train uniquement.
_CLEAN_SIZE = 600
clean_alphabets = {}
if any(isinstance(c.get("alphabet"), str) for c in CONFIGS):
    _freq = Counter()
    for texts in train_by_lang.values():
        for t in texts:
            _freq.update(ch for ch in t if not ch.isspace())
    _ranked = [ch for ch, _ in _freq.most_common()]

    def _script_kept(ch, keep_prefixes):
        cat = unicodedata.category(ch)
        if cat[0] in ("N", "P", "S", "M"):
            return True
        return unicodedata.name(ch, "").startswith(keep_prefixes)

    for _mode, _prefixes in (("clean_v1", ("LATIN", "ETHIOPIC", "ARABIC")),
                             ("clean_v2", ("LATIN", "ETHIOPIC"))):
        clean_alphabets[_mode] = [ch for ch in _ranked if _script_kept(ch, _prefixes)][:_CLEAN_SIZE]
    _top500 = set(_ranked[:500])
    _ethio_extra = [ch for ch in _ranked
                    if ch not in _top500 and unicodedata.name(ch, "").startswith("ETHIOPIC")]
    clean_alphabets["gez_topup"] = _ranked[:500] + _ethio_extra
    for _mode, _alpha in clean_alphabets.items():
        _scripts = Counter(unicodedata.name(ch, "UNKNOWN").split()[0] for ch in _alpha)
        print(f"Alphabet {_mode} : {len(_alpha)} caractères — {_scripts.most_common(6)}")

# Tirage : permet de lancer un sous-ensemble  ->  RUN_CONFIGS = {"c4-wssplit-mf10-alpha"}
RUN_CONFIGS = None            # None = toutes les configurations
selected = [c for c in CONFIGS if RUN_CONFIGS is None or c["name"] in RUN_CONFIGS]
print(f"\n{len(selected)} configuration(s) à entraîner :")
for c in selected:
    print(f"  - {c['name']:32s} {c['note']}")

## 5. Entraînement + évaluation de chaque configuration

Pour chaque configuration : entraînement sur `train`, puis évaluation **sur `validation`** avec la
métrique officielle (score, fertility par langue, UNK, **verdict guardrail**).

⏱️ Durée indicative dans Colab : ~30 s par config BPE, ~5 min pour l'unigram — les 17 configs prennent environ 45–75 min (mesure 24k lignes incluse).
Utiliser `MAX_TRAIN_DOCS = 60_000` dans la cellule 2 pour un pré-balayage rapide, puis relancer les
2–3 meilleures en données complètes.

In [ ]:
# =============================================================================
# 5. Balayage : entraînement + évaluation officielle
# =============================================================================
def corpus_iterator(train_by_lang, boost=None, log_every=50_000):
    """Itère les textes multilingues en round-robin (équilibré) avec sur-échantillonnage."""
    boost = boost or {}
    iters = {l: iter(texts) for l, texts in train_by_lang.items()}
    active = list(train_by_lang)
    i = 0
    while active:
        for lang in list(active):
            try:
                text = next(iters[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text
                i += 1
                if log_every and i % log_every == 0:
                    print(f"    ... {i:,} textes fournis")


def strip_byte_added_tokens(tokenizer):
    """Rend les tokens byte ordinaires (forme des tokenizers Llama-2).

    Le trainer les ajoute via special_tokens (seul moyen mesuré de les faire entrer dans le
    vocabulaire du modèle) : on retire leur entrée `added_tokens`, ils restent des tokens du
    modèle — byte fallback intact, et decode(skip_special_tokens=True) ne les jette plus.

    Quirk mesuré (tokenizers 0.22.1) : UnigramTrainer ne conserve PAS byte_fallback
    (le modèle réapprend avec le flag à False, et le constructeur sans vocab ne le
    sérialise pas) — sans la réactivation ci-dessous, l'unigram HF réémet des [UNK].
    BPE n'est pas affecté (le flag survit à l'entraînement) : l'assignation est neutre.
    """
    payload = json.loads(tokenizer.to_str())
    payload["added_tokens"] = [t for t in payload["added_tokens"]
                               if not is_byte_token(t["content"])]
    if isinstance(payload.get("model"), dict) and "byte_fallback" in payload["model"]:
        payload["model"]["byte_fallback"] = True
    return Tokenizer.from_str(json.dumps(payload))


SP_METASPACE = "\u2581"      # marqueur d'espace de SentencePiece (pas U+2581 litteral dans le code)


def build_sentencepiece_tokenizer(cfg):
    """Entraîne SentencePiece puis le convertit en tokenizer.json HuggingFace.

    Conversion mesurée localement : les pièces SentencePiece (dont les 256 <0xXX> du
    byte_fallback) deviennent un modèle Unigram, avec un pré-tokeniseur/décodeur Metaspace
    — le checker officiel valide le fichier obtenu (valid=True, vocab 10 000).
    """
    import sentencepiece as spm

    model_type = "unigram" if cfg["model"] == "sp_unigram" else "bpe"
    sp_dir = OUTPUT_ROOT / "sentencepiece"
    sp_dir.mkdir(parents=True, exist_ok=True)
    prefix = sp_dir / cfg["name"]
    spm.SentencePieceTrainer.train(
        sentence_iterator=corpus_iterator(train_by_lang, cfg.get("boost")),
        model_prefix=str(prefix), vocab_size=VOCAB_SIZE, model_type=model_type,
        byte_fallback=True,
        character_coverage=cfg.get("character_coverage", 0.9995),
        normalization_rule_name="identity",      # aucun NFKC/NFKD, aucune conversion ASCII
        # hard_vocab_limit=False : mesuré localement, sinon SentencePiece peut boucler sans fin
        # quand l'alphabet est grand et le corpus insuffisant pour remplir 10 000 pièces.
        hard_vocab_limit=False,
        num_threads=4, minloglevel=2)
    sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
    vocab = [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())]
    tokenizer = Tokenizer(Unigram(vocab=vocab, unk_id=sp.unk_id(), byte_fallback=True))
    tokenizer.pre_tokenizer = Metaspace(replacement=SP_METASPACE, prepend_scheme="always",
                                        split=True)
    tokenizer.decoder = Sequence([
        ByteFallback(),
        MetaspaceDec(replacement=SP_METASPACE, prepend_scheme="always"),
    ])
    print(f"    SentencePiece {model_type} : {sp.get_piece_size():,} pièces "
          f"(dont byte fallback), converti en Unigram + Metaspace")
    return tokenizer


def build_tokenizer(cfg):
    """Construit et entraîne un tokenizer selon la configuration (utilise train seulement)."""
    if cfg["model"] in ("sp_unigram", "sp_bpe"):
        return build_sentencepiece_tokenizer(cfg)

    byte_fallback = bool(cfg.get("byte_fallback", False))
    if cfg["model"] == "bpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]", byte_fallback=byte_fallback))
    elif cfg["model"] == "wordpiece":
        tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
    else:
        # NOTE : byte_fallback=True au constructeur est perdu par UnigramTrainer
        # (mesuré) — le flag est réactivé dans strip_byte_added_tokens() après
        # entraînement. Ne PAS ajouter le paramètre ici (trompeur).
        tokenizer = Tokenizer(Unigram())
    # Normaliseur (EXP-009 : nfc / nfkc / nfc_lower / nfkc_lower ; défaut nfc)
    norm = cfg.get("normalizer", "nfc")
    if norm == "nfc":
        tokenizer.normalizer = NFC()
    elif norm == "nfkc":
        tokenizer.normalizer = NFKC()
    elif norm == "nfc_lower":
        tokenizer.normalizer = NormSequence([NFC(), Lowercase()])
    elif norm == "nfkc_lower":
        tokenizer.normalizer = NormSequence([NFKC(), Lowercase()])
    else:
        raise ValueError(f"normaliseur inconnu : {norm}")

    if cfg["pre"] == "whitespace":
        tokenizer.pre_tokenizer = Whitespace()
    elif cfg["pre"] == "whitespace_split":
        tokenizer.pre_tokenizer = WhitespaceSplit()
    elif cfg["pre"] == "whitespace_split_punct":
        # EXP-008 : mots propres — la ponctuation devient un token à part (behavior=isolated),
        # les merges apprennent les vraies fréquences de mots au lieu de formes « mot, ».
        tokenizer.pre_tokenizer = PreSequence([WhitespaceSplit(),
                                               Split(Regex(r"\p{P}"), behavior="isolated")])
    elif cfg["pre"] == "whitespace_split_punct_digits":
        # EXP-009 : comme ponctuation isolée + nombres isolés (MorphBPE-light :
        # aucun merge ne traverse ponctuation ou chiffres).
        tokenizer.pre_tokenizer = PreSequence([WhitespaceSplit(),
                                               Split(Regex(r"\p{P}"), behavior="isolated"),
                                               Split(Regex(r"\d+"), behavior="isolated")])
    elif cfg["pre"] == "byte_level":
        tokenizer.pre_tokenizer = ByteLevelPre(add_prefix_space=False, use_regex=True)
        tokenizer.decoder = ByteLevelDec()

    if cfg["alphabet"] == "bytes":
        alphabet = ByteLevelPre.alphabet()
    elif cfg["alphabet"] is True:
        alphabet = train_alphabet
    elif cfg.get("limit_alphabet") and cfg["model"] == "unigram":
        # EXP-010 : top-N explicite (UnigramTrainer ignore limit_alphabet).
        alphabet = top_alphabets[cfg["limit_alphabet"]]
    elif isinstance(cfg.get("alphabet"), str) and cfg["alphabet"] in clean_alphabets:
        # EXP-202 : alphabet nettoyé explicite (pas de limit_alphabet : la liste EST
        # l'alphabet ; les caractères écartés passent en byte fallback, zéro [UNK]).
        alphabet = clean_alphabets[cfg["alphabet"]]
    else:
        alphabet = None

    kwargs = dict(vocab_size=VOCAB_SIZE, min_frequency=cfg["min_freq"],
                  special_tokens=["[UNK]"] + (BYTE_TOKENS if byte_fallback else []))
    if alphabet:
        kwargs["initial_alphabet"] = alphabet
    # limit_alphabet : ne garder que les N caractères les plus fréquents du train ; les
    # caractères écartés sont couverts par le byte fallback (aucun [UNK]).
    # limit_alphabet : BPE/WordPiece uniquement (UnigramTrainer l'ignore — voir
    # initial_alphabet explicite ci-dessus ; le lui passer afficherait un warning).
    if cfg.get("limit_alphabet") and cfg["model"] in ("bpe", "wordpiece"):
        kwargs["limit_alphabet"] = cfg["limit_alphabet"]
    if cfg["model"] == "bpe":
        trainer = BpeTrainer(**kwargs)
    elif cfg["model"] == "wordpiece":
        trainer = WordPieceTrainer(**kwargs)
    else:
        kwargs.pop("min_frequency", None)      # UnigramTrainer n'accepte pas min_frequency
        kwargs["unk_token"] = "[UNK]"
        trainer = UnigramTrainer(**kwargs)

    tokenizer.train_from_iterator(
        corpus_iterator(train_by_lang, cfg.get("boost")), trainer=trainer)

    if byte_fallback:
        tokenizer.decoder = ByteFallback()
        tokenizer = strip_byte_added_tokens(tokenizer)
        print(f"    byte_fallback : {len(BYTE_TOKENS)} tokens <0xXX> dans le vocabulaire "
              f"(total {tokenizer.get_vocab_size(with_added_tokens=True):,}), "
              f"added_tokens restants = {len(json.loads(tokenizer.to_str())['added_tokens'])}")
    return tokenizer


results = []
for n, cfg in enumerate(selected, start=1):
    print(f"\n{'='*84}\n[{n}/{len(selected)}] {cfg['name']} — {cfg['note']}\n{'='*84}")
    t0 = time.time()
    try:
        tok = build_tokenizer(cfg)
    except Exception as exc:                     # une config qui échoue ne bloque pas le reste
        print(f"  CONFIGURATION IGNORÉE : {type(exc).__name__} : {exc}")
        continue
    train_seconds = time.time() - t0

    fertility, unk_rate, tokens, words, lossy, unk_total = measure(val_rows, tok)
    quality = measure_quality(tok, quality_rows)
    score = competition_score(fertility, unk_rate)
    raw, budget, breaches = guardrail(fertility)
    penalised = penalised_scores(fertility, unk_rate)
    vocab_size = tok.get_vocab_size(with_added_tokens=True)

    results.append(dict(
        name=cfg["name"], note=cfg["note"], config=cfg,
        vocab_size=vocab_size, train_seconds=train_seconds,
        score=score, fertility=fertility, unk_rate=unk_rate, penalised=penalised,
        tokens=tokens, words=words, unk_total=unk_total, lossy_rows=lossy,
        raw_scored=raw, guardrail_budget=budget, guardrail_breaches=breaches,
        guardrail_pass=not breaches, tokenizer=tok,
        quality_mean=quality["mean"], quality=quality,
    ))
    flag = "GUARDRAIL OK" if not breaches else f"GUARDRAIL FAIL {breaches}"
    print(f"  score={score:.4f} | jacc={quality['mean']:.4f} | vocab={vocab_size} | UNK={unk_total} | "
          f"lossy={lossy:,} | {flag} | {train_seconds/60:.1f} min")

print("\nBalayage terminé.")

## 6. Résultats classés + reproductibilité

Tableau trié par score officiel (plus bas = meilleur), avec le verdict guardrail et le
Jaccard de chaque configuration. Les calibrateurs c30/c32 doivent redonner la phase 1
à ±0.002, sinon le run n'est pas interprétable. La sélection ne porte que sur les
NOUVELLES configs : à ±0.01 du meilleur brut, on retient le Jaccard maximal.


In [ ]:
# =============================================================================
# 6. Tableau comparatif + classement
# =============================================================================
rows = []
for r in sorted(results, key=lambda x: x["score"]):
    rows.append({
        "config": r["name"],
        "Score ↓": round(r["score"], 4),
        "Jacc ↑": round(r["quality_mean"], 4),
        "Guardrail": "PASS" if r["guardrail_pass"] else "FAIL",
        "Hausa": round(r["penalised"]["ha"], 4),
        "Swahili": round(r["penalised"]["sw"], 4),
        "Yoruba": round(r["penalised"]["yo"], 4),
        "Amharic": round(r["penalised"]["am"], 4),
        "UNK": r["unk_total"],
        "lossy": f"{r['lossy_rows']:,}",
        "vocab": r["vocab_size"],
        "en": round(r["fertility"]["en"], 4),
        "fr": round(r["fertility"]["fr"], 4),
        "budget": round(r["guardrail_budget"], 4),
    })
sweep_df = pd.DataFrame(rows)
print("Score officiel = moyenne des scores (ha, sw, yo, am). Lower is better.")
print("Guardrail = fertility(en) et fertility(fr) <= 1.15 x moyenne brute des 4 langues notées.\n")
print(sweep_df.to_string(index=False))

# --- Reproductibilité : les calibrateurs doivent redonner la phase 1 (±0.002) ---
print("\n--- Reproductibilité (reconstruction depuis zéro, §15) ---")
repro_ok = True
for _name, _ref_s, _ref_j in [("c30-lower-alph500-bf", C30_REF_SCORE, C30_REF_JACCARD),
                              ("c32-lower-alph1000-bf-b3", REF_SCORE, REF_JACCARD)]:
    _r = next((x for x in results if x["name"] == _name), None)
    if _r is None:
        print(f"  {_name}: NON EXÉCUTÉ (RUN_CONFIGS partiel ?)")
        repro_ok = False
    else:
        _ds, _dj = _r["score"] - _ref_s, _r["quality_mean"] - _ref_j
        _ok = abs(_ds) <= REPRO_TOL
        repro_ok = repro_ok and _ok
        print(f"  {_name}: score {_r['score']:.4f} (réf {_ref_s:.4f}, écart {_ds:+.4f}) "
              f"jaccard {_r['quality_mean']:.4f} (réf {_ref_j:.4f}) → {'OK' if _ok else 'ÉCART !'}")

# --- Sélection parmi les NOUVELLES configs (calibrateurs exclus) ---
new_results = [r for r in results if r["name"] not in CALIBRATORS]
new_valid = [r for r in new_results if r["guardrail_pass"]]
new_raw = min(new_valid, key=lambda r: r["score"]) if new_valid else None
if new_raw:
    _short = [r for r in new_valid if r["score"] <= new_raw["score"] + QUALITY_TIEBAND]
    best = min(_short, key=lambda r: (-r["quality_mean"], r["score"]))
else:
    best = None

if best:
    gain = REF_SCORE - best["score"]
    print(f"\nMEILLEUR SCORE BRUT PHASE 2 : {new_raw['name']} -> {new_raw['score']:.4f} "
          f"(jaccard {new_raw['quality_mean']:.4f})")
    if best["name"] != new_raw["name"]:
        print(f"(la qualité départage : {best['name']} à {best['score']:.4f} "
              f"(+{best['score'] - new_raw['score']:.4f}) avec jaccard {best['quality_mean']:.4f})")
    print(f"SÉLECTION PHASE 2 (score + qualité) : {best['name']}")
    print(f"  score {best['score']:.4f}  (vs réf {REF_SCORE:.4f} : {gain:+.4f} soit {100*gain/REF_SCORE:+.2f} %)")
    print("  qualité : jaccard moyen {:.4f} (réf {:.4f}) | ".format(best["quality_mean"], REF_JACCARD) +
          " | ".join(f"{v} {best['quality'][v]:.4f}" for v in QUALITY_VARIANTS))
    print(f"  guardrail : budget {best['guardrail_budget']:.4f} | en {best['fertility']['en']:.4f} "
          f"(marge {best['guardrail_budget']-best['fertility']['en']:+.4f}) | fr {best['fertility']['fr']:.4f} "
          f"(marge {best['guardrail_budget']-best['fertility']['fr']:+.4f})")
    for l in LANGUAGES:
        print(f"    {l}: fertility {best['fertility'][l]:.4f} | unk_rate {best['unk_rate'][l]:.6f} | score {best['penalised'][l]:.4f}")
else:
    print("Aucune nouvelle configuration ne passe le guardrail : revoir la matrice.")


## 7. Analyse automatique : quel levier a fonctionné ?

Comparaison des axes testés pour comprendre **pourquoi** une configuration gagne.

In [ ]:
# =============================================================================
# 7. Analyse des leviers
# =============================================================================
base = next((r for r in results if r["name"] == REF_NAME), None)
print("Effet des leviers (score relatif au rerun c32) :\n")
for r in sorted(results, key=lambda x: x["score"]):
    rel = "" if base is None else f"{r['score'] - base['score']:+.4f}"
    print(f"  {r['name']:34s} {r['score']:.4f}  ({rel})")

print("\nLecture des UNK :")
for r in results:
    print(f"  {r['name']:34s} UNK total={r['unk_total']:>6} | " +
          " ".join(f"{l}:{r['unk_rate'][l]*100:.3f}%" for l in LANGUAGES))

print("\nLecture de la fertility par langue (brute) :")
print(f"  {'config':34s} " + " ".join(f"{l:>8s}" for l in LANGUAGES))
for r in results:
    print(f"  {r['name']:34s} " + " ".join(f"{r['fertility'][l]:>8.4f}" for l in LANGUAGES))
print("\nLecture de la qualité (Jaccard : robustesse aux variantes, plus haut = meilleur) :")
print(f"  {'config':34s} {'jacc':>8s} " + " ".join(f"{v:>11s}" for v in QUALITY_VARIANTS))
for r in sorted(results, key=lambda x: x["score"]):
    print(f"  {r['name']:34s} {r['quality_mean']:>8.4f} " +
          " ".join(f"{r['quality'][v]:>11.4f}" for v in QUALITY_VARIANTS))


## 7bis. Diagnostic qualité du tokenizer sélectionné

Chiffre les gisements restants : tokens morts, merges avec ponctuation collée, doublons de
casse, mots fréquents fragmentés — plus les parts de mots capitalisés / à ponctuation collée /
avec chiffres qui motivent les leviers P1/P2 (lowercase, frontières propres).


In [ ]:
# =============================================================================
# 7bis. Diagnostic qualité du tokenizer sélectionné (lecture seule)
# =============================================================================
import unicodedata as _ud

diag_tok = best["tokenizer"]
diag_vocab = diag_tok.get_vocab()
print(f"Diagnostic de `{best['name']}` (vocab {len(diag_vocab):,}) — score {best['score']:.4f}, "
      f"jaccard {best['quality_mean']:.4f}\n")

# 1. Tokens morts : jamais émis sur validation (hors [UNK]/<unk> et bytes <0xXX>)
used_ids = set()
for start in range(0, len(val_rows), 2048):
    batch = val_rows[start:start + 2048]
    for enc in diag_tok.encode_batch([t for _, t in batch], add_special_tokens=False):
        used_ids.update(enc.ids)


def _is_byte(tok):
    return len(tok) == 6 and tok.startswith("<0x") and tok.endswith(">")


_special_ids = {i for tok in ("[UNK]", "<unk>") for i in [diag_vocab.get(tok)] if i is not None}
_byte_ids = {i for tok, i in diag_vocab.items() if _is_byte(tok)}
dead = set(diag_vocab.values()) - used_ids - _special_ids - _byte_ids
print(f"1. tokens appris jamais émis sur validation : {len(dead):,} "
      f"(soit {100 * len(dead) / max(len(diag_vocab), 1):.1f} % du vocab)")

# 2. Merges avec ponctuation collée (pistes « mot, » — cf. MorphBPE-light)
punct_toks = [t for t in diag_vocab if len(t) >= 2 and not _is_byte(t)
              and any(_ud.category(c).startswith("P") for c in t)]
print(f"2. tokens appris (≥2 chars) contenant de la ponctuation : {len(punct_toks):,}")

# 3. Doublons de casse (« The »/« the » — gisement du lowercase)
_by_lower = defaultdict(list)
for t in diag_vocab:
    if not _is_byte(t):
        _by_lower[t.lower()].append(t)
_dupes = {k: v for k, v in _by_lower.items() if len(v) > 1}
print(f"3. groupes de doublons de casse : {len(_dupes):,} "
      f"(places potentiellement libérables par lowercase)")
if _dupes:
    print(f"   exemples : {list(_dupes.values())[:6]}")

# 4. Top-30 mots fragmentés par langue notée (1 token = idéal)
print("4. top-30 mots par langue notée :")
for lang in SCORED_LANGUAGES:
    texts = [t for l, t in val_rows if l == lang]
    wc = Counter(w for t in texts for w in t.split())
    top = [w for w, _ in wc.most_common(30)]
    frag = [(w, diag_tok.encode(w, add_special_tokens=False).tokens) for w in top]
    single = sum(1 for _, toks in frag if len(toks) == 1)
    examples = [(w, toks) for w, toks in frag if len(toks) > 1][:3]
    print(f"   {lang}: {single}/30 en 1 token ; fragmentés ex : {examples}")

# 5. Parts motivant P1/P2 (indépendant du tokenizer — corpus de validation)
print("5. structure des mots (validation) :")
for lang in LANGUAGES:
    texts = [t for l, t in val_rows if l == lang]
    words = [w for t in texts for w in t.split()]
    cap = sum(1 for w in words if w[:1].isupper())
    punct = sum(1 for w in words if w and _ud.category(w[-1]).startswith("P"))
    digit = sum(1 for w in words if any(c.isdigit() for c in w))
    print(f"   {lang}: {len(words):,} mots — capitalisés {cap / len(words):.1%} | "
          f"ponct collée {punct / len(words):.1%} | avec chiffres {digit / len(words):.1%}")


## 8. Sauvegarde du meilleur phase 2 + rapports

Le tokenizer gagnant est écrit dans `models/phase2_best/tokenizer.json` (la référence
`models/optimized_c32-...` reste intacte) et les rapports dans
`reports/optimization_phase2.{json,md}` (le sweep phase 1 reste intact).


In [ ]:
# =============================================================================
# 8. Sauvegarde du meilleur + rapports JSON/Markdown
# =============================================================================
best_model_dir = MODEL_DIR / "phase2_best"
best_model_dir.mkdir(parents=True, exist_ok=True)
best_tokenizer_path = best_model_dir / "tokenizer.json"
best["tokenizer"].save(str(best_tokenizer_path))
print("Meilleur tokenizer sauvegardé :", best_tokenizer_path,
      f"({best_tokenizer_path.stat().st_size:,} octets)")

# Copie "candidate" à la racine pour le checker officiel
candidate_path = OUTPUT_ROOT / "tokenizer.json"
shutil.copy2(best_tokenizer_path, candidate_path)
print("Candidat pour le checker :", candidate_path)

report = {
    "experiment": "optimization_phase2",
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "train_docs_used": {l: len(v) for l, v in train_by_lang.items()},
                "validation_rows": len(val_rows)},
    "official_rules": {
        "scored_languages": list(SCORED_LANGUAGES),
        "context_languages": list(CONTEXT_LANGUAGES),
        "guardrail_ratio": CONTEXT_FERTILITY_RATIO,
        "unknown_penalty": UNKNOWN_PENALTY,
        "max_vocab_size": MAX_VOCAB_SIZE,
        "required_tokenizers_version": REQUIRED_TOKENIZERS_VERSION,
        "word_definition": "len(text.split())",
    },
    "settings": {"vocab_size": VOCAB_SIZE, "max_train_docs": MAX_TRAIN_DOCS,
                "quality": {"samples_per_lang": QUALITY_SAMPLES_PER_LANG, "seed": QUALITY_SEED,
                            "variants": list(QUALITY_VARIANTS),
                            "selection_tieband": QUALITY_TIEBAND}},
    "baseline_reference_score": BASELINE_REFERENCE_SCORE,
    "results": [
        {k: v for k, v in r.items() if k not in ("tokenizer",)}
        for r in sorted(results, key=lambda x: x["score"])
    ],
    "best": {"name": best["name"], "score": best["score"], "config": best["config"],
             "guardrail_pass": best["guardrail_pass"], "model_path": str(best_tokenizer_path),
             "quality_mean": best["quality_mean"], "quality": best["quality"]},
    "best_raw_score": {"name": new_raw["name"], "score": new_raw["score"],
                       "quality_mean": new_raw["quality_mean"]},
    "phase1_reference": {"name": REF_NAME, "score": REF_SCORE, "jaccard": REF_JACCARD},
    "replacement_thresholds": {"min_gain": REPLACE_MIN_GAIN, "min_margin": REPLACE_MIN_MARGIN,
                               "jaccard_tolerance": REPLACE_JACC_TOL},
    "environment": {"tokenizers": __import__("tokenizers").__version__,
                    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
}
(REPORT_DIR / "optimization_phase2.json").write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                                   encoding="utf-8")

# --- Markdown ---------------------------------------------------------------
md = ["# Phase 2 — Optimisation ciblée (score + qualité)", "",
      f"*Dataset `{DATASET_NAME}` @ `{DATASET_REVISION}` — train utilisé : "
      f"{sum(len(v) for v in train_by_lang.values()):,} textes, validation : {len(val_rows):,} lignes.*", "",
      "## Classement", "",
      "| Config | Score ↓ | Jacc ↑ | Guardrail | Hausa | Swahili | Yoruba | Amharic | UNK | lossy | vocab | en | fr | budget |",
      "|---|---:|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
for r in sorted(results, key=lambda x: x["score"]):
    md.append(f"| `{r['name']}` | {r['score']:.4f} | {r['quality_mean']:.4f} | {'PASS' if r['guardrail_pass'] else 'FAIL'} | "
              f"{r['penalised']['ha']:.4f} | {r['penalised']['sw']:.4f} | {r['penalised']['yo']:.4f} | "
              f"{r['penalised']['am']:.4f} | {r['unk_total']} | {r['lossy_rows']:,} | {r['vocab_size']} | "
              f"{r['fertility']['en']:.4f} | {r['fertility']['fr']:.4f} | {r['guardrail_budget']:.4f} |")
md += ["", "## Fertility brute par langue", "",
       "| Config | " + " | ".join(LANGUAGES) + " |", "|---|" + "---:|" * len(LANGUAGES)]
for r in sorted(results, key=lambda x: x["score"]):
    md.append(f"| `{r['name']}` | " + " | ".join(f"{r['fertility'][l]:.4f}" for l in LANGUAGES) + " |")
md += ["", "## Décision", "",
       f"- Baseline originelle : **{BASELINE_REFERENCE_SCORE:.4f}**",
       f"- Référence phase 1 : `{REF_NAME}` — {REF_SCORE:.4f} (jaccard {REF_JACCARD:.4f})",
       f"- Meilleur score brut phase 2 : `{new_raw['name']}` — {new_raw['score']:.4f} "
       f"(jaccard {new_raw['quality_mean']:.4f})",
       f"- **Sélection phase 2 (score + qualité) : `{best['name']}` — score {best['score']:.4f}** "
       f"(vs réf {REF_SCORE - best['score']:+.4f}, jaccard {best['quality_mean']:.4f})",
       f"- Guardrail : budget {best['guardrail_budget']:.4f}, en {best['fertility']['en']:.4f}, "
       f"fr {best['fertility']['fr']:.4f} → {'PASS' if best['guardrail_pass'] else 'FAIL'}",
       f"- Modèle : `{best_tokenizer_path}`", "",
       "## Configurations testées", ""]
for r in results:
    md.append(f"- `{r['name']}` — {r['note']} — score {r['score']:.4f} — "
              f"jaccard {r['quality_mean']:.4f} — guardrail {'PASS' if r['guardrail_pass'] else 'FAIL'}")
md.append("")
(REPORT_DIR / "optimization_phase2.md").write_text("\n".join(md), encoding="utf-8")
print("Rapports :", REPORT_DIR / "optimization_phase2.json", "|", REPORT_DIR / "optimization_phase2.md")

## 9. Vérification avec le **checker officiel** du challenge

On télécharge `starter/utils.py` du dépôt officiel et on exécute `profile_submission` sur le
tokenizer gagnant, avec le split de validation comme données. C'est **le même code** que celui
utilisé pour valider les soumissions.

In [ ]:
# =============================================================================
# 9. Checker officiel (starter/utils.py du dépôt du challenge)
# =============================================================================
OFFICIAL_UTILS_URL = ("https://raw.githubusercontent.com/aims-ai-research-foundations/"
                      "airf-multilingual-tokenizer-challenge/main/starter/utils.py")
utils_path = OUTPUT_ROOT / "utils.py"
if not utils_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(OFFICIAL_UTILS_URL, utils_path)
        print("utils.py officiel téléchargé")
    except Exception as exc:
        print("Téléchargement impossible :", exc)

official_report = None
if utils_path.exists():
    import importlib, sys
    sys.path.insert(0, str(OUTPUT_ROOT))
    import utils as official_utils
    importlib.reload(official_utils)
    official_report = official_utils.profile_submission(
        candidate_path, data=pd.DataFrame(val_rows, columns=["language", "text"]))
else:
    print("Checker officiel indisponible — utilisation des contrôles intégrés.")
    print(json.dumps(validate_tokenizer_file(candidate_path), indent=2, ensure_ascii=False))

# Contrôles intégrés complémentaires (équivalents à competition/validation.py)
checks = validate_tokenizer_file(candidate_path)
print("\nContrôles officiels intégrés :", checks["checks"])
print("Erreurs :", checks["errors"] or "aucune")
print("Langues non lossless (round-trip) :", checks["lossy_languages"] or "aucune (lossless)")

## 10. Décision finale (§18 + §19) — AUCUNE écriture dans `submissions/`

Imprime le rapport final et la recommandation :

- **OPTION B (REPLACE)** — gain ≥ 0.005, jaccard tenu, UNK = 0, marges ≥ 0.04, checker READY ;
- **OPTION C (CONTINUE)** — une sonde frontière gagne encore → phase 2B justifiée ;
- **OPTION A (KEEP)** — sinon : c32 reste la référence.

En cas d'OPTION B, la promotion vers `submissions/` se fait MANUELLEMENT après
publication des artefacts (cellule 12) — jamais automatiquement.


In [ ]:
# =============================================================================
# 10. Décision finale (§18 rapport + §19 A/B/C) — ne touche PAS submissions/
# =============================================================================
# La promotion éventuelle vers submissions/ se fait MANUELLEMENT, après publication
# des artefacts (cellule 12) : ce notebook ne modifie jamais la soumission valide.
print("=" * 70)
print("AIRF TOKENIZER — PHASE 2 : RAPPORT FINAL")
print("=" * 70)
margin_en = best["guardrail_budget"] - best["fertility"]["en"]
margin_fr = best["guardrail_budget"] - best["fertility"]["fr"]
gain = REF_SCORE - best["score"]
if official_report is not None:
    checker_ready = bool(official_report.get("valid"))
    checker_src = "officiel (starter/utils.py)"
else:
    checker_ready = bool(checks["valid"])
    checker_src = "intégré (équivalence vérifiée)"
print(f"""
Previous baseline : {BASELINE_REFERENCE_SCORE:.4f}
Previous best raw : {C30_REF_SCORE:.4f} (c30-lower-alph500-bf)
Previous selected : {REF_SCORE:.4f} ({REF_NAME}, jaccard {REF_JACCARD:.4f})
New best          : {best['score']:.4f} ({best['name']}, jaccard {best['quality_mean']:.4f})
Improvement       : {gain:+.4f} ({100 * gain / REF_SCORE:+.2f} % vs phase 1)
Guardrail         : {'PASS' if best['guardrail_pass'] else 'FAIL'} \
(marges en {margin_en:+.4f} / fr {margin_fr:+.4f})
Official checker  : {'READY FOR SUBMISSION' if checker_ready else 'NOT READY'} ({checker_src})
""")
print("Par langue (fertility | unk_rate | score) :")
for l in LANGUAGES:
    print(f"  {l}: fertility {best['fertility'][l]:.4f} | "
          f"unk_rate {best['unk_rate'][l]:.6f} | score {best['penalised'][l]:.4f}")
print()
decision_checks = [
    ("gain significatif (≥ 0.005)", gain >= REPLACE_MIN_GAIN),
    ("jaccard acceptable (≥ réf − 0.005)",
     best["quality_mean"] >= REF_JACCARD - REPLACE_JACC_TOL),
    ("UNK == 0", best["unk_total"] == 0),
    ("guardrail PASS + marges ≥ 0.04",
     best["guardrail_pass"] and min(margin_en, margin_fr) >= REPLACE_MIN_MARGIN),
    ("checker READY", checker_ready),
]
for _label, _ok in decision_checks:
    print(f"  [{'OK' if _ok else '--'}] {_label}")
print()
_FRONTIER_PROBES = {"c42-lower-alph500-b4", "c45-lower-alph500-am6"}
if all(_ok for _, _ok in decision_checks):
    print("DÉCISION : OPTION B — REPLACE WITH PHASE 2 TOKENIZER")
    print("Le tokenizer phase 2 est réellement supérieur : il peut promouvoir submissions/.")
elif best["name"] in _FRONTIER_PROBES and gain > 0:
    print("DÉCISION : OPTION C — CONTINUE OPTIMIZATION")
    print("La frontière (boost) s'améliore encore : une phase 2B (boost+, allocation amharique)")
    print("est justifiée avant toute promotion.")
elif gain > 0.002:
    print("DÉCISION : OPTION A — KEEP CURRENT TOKENIZER (gain dans le bruit)")
    print("Pas de gain significatif : c32 reste la référence.")
else:
    print("DÉCISION : OPTION A — KEEP CURRENT TOKENIZER")
    print("Aucun gain : c32 reste la référence.")
print()
print("Artefacts phase 2 :", best_model_dir / "tokenizer.json")
print("Pour promouvoir (OPTION B uniquement) : publiez via la cellule 12, puis demandez")
print("la promotion — submissions/ n'est JAMAIS modifié automatiquement par ce notebook.")


## 11. Prochaines étapes

1. **Publier** (cellule 12) : `models/phase2_best/` + `reports/optimization_phase2.*`
   vers la branche de travail (`main` intacte, `submissions/` intact).
2. **Si OPTION B** : demander la promotion — le dossier `submissions/maick-dane-nkou/`
   sera régénéré (tokenizer + metadata + README + notebook 04) puis la PR officielle.
3. **Si OPTION C** : phase 2B (boost extrême amharique, allocation latin/guèze).
4. **Si OPTION A** : c32 reste la soumission ; ce run documente ce qui a été épuisé.

Rappels : `tokenizer.json` ≤ 20 MiB, vocab ≤ 10 000, `tokenizers==0.22.1`,
entraînement sur `train` uniquement, test caché = garder des marges (≥ 0.04).


In [ ]:
# =============================================================================
# 11. Récapitulatif final
# =============================================================================
print("Phase 2 terminée.\n")
print(f"Baseline de référence : {BASELINE_REFERENCE_SCORE:.4f}")
print(f"Référence phase 1 : {REF_NAME} -> {REF_SCORE:.4f} (jaccard {REF_JACCARD:.4f})")
print(f"Meilleur brut phase 2 : {new_raw['name']} -> {new_raw['score']:.4f} (jaccard {new_raw['quality_mean']:.4f})")
print(f"Sélection phase 2 : {best['name']} -> {best['score']:.4f} "
      f"(vs réf {REF_SCORE - best['score']:+.4f}, jaccard {best['quality_mean']:.4f})")
print(f"Guardrail : {'PASS' if best['guardrail_pass'] else 'FAIL'}")
print()
print("Artefacts :")
print(f"  - {best_tokenizer_path}")
print(f"  - {REPORT_DIR / 'optimization_phase2.json'}")
print(f"  - {REPORT_DIR / 'optimization_phase2.md'}")
print("  - submissions/ : INTACT (promotion manuelle uniquement, voir cellule 10)")
if official_report is not None:
    print("\nChecker officiel :", "READY FOR SUBMISSION" if official_report.get("valid")
          else f"NOT READY -> {official_report.get('errors')}")

## 12. Publier les artefacts sur GitHub (script + token)

La phase 2 a produit `models/phase2_best/` et `reports/optimization_phase2.{json,md}`
(`submissions/` n'est pas modifié par ce notebook). La cellule suivante **écrit le script de publication** (contenu identique à
`scripts/push_artifacts_to_github.py` du dépôt) et la dernière l'**exécute dans le processus du
notebook** : un **champ masqué** s'affiche pour coller le token GitHub (portée `repo`).

`--include-submissions` publie aussi le dossier de soumission. Le script refuse de publier des
artefacts non conformes (run sur données synthétiques) et masque le token dans toutes les sorties.

### Cellule « script de publication »

La cellule suivante **écrit le script** `push_artifacts_to_github.py` dans le répertoire courant
(contenu identique à `scripts/push_artifacts_to_github.py` du dépôt), et celle d'après **l'exécute
dans le processus du notebook** — indispensable pour que le **champ masqué Colab** fonctionne et
pour que le script voie les Secrets Colab.

Le token est demandé par saisie masquée (ou lu dans le secret Colab `GITHUB_TOKEN` s'il existe).
Il n'est jamais affiché, jamais écrit sur disque, jamais commité.

Ce que le script publie : `models/**`, `reports/**` (+ `submissions/**` avec `--include-submissions`),
sur la branche `arena/01a092c6-tokenizer` (`main` reste intacte).

In [ ]:
%%writefile push_artifacts_to_github.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Publie les artefacts du challenge vers votre dépôt GitHub (méthode 2 : token).

Le token est fourni par saisie **masquée** (recommandé), par variable
d'environnement, ou par le secret Colab ``GITHUB_TOKEN``. Il n'est **jamais**
affiché, jamais écrit sur disque, jamais commité : toutes les sorties passent par
``redact()``.

Ce qui est publié (par défaut) :
    models/**      tokenizer(s) entraîné(s)
    reports/**     rapports JSON / Markdown
    submissions/** (avec --include-submissions) dossier de soumission

Exemples
--------
Colab — recommandé (dans une cellule Python, champ masqué actif) :
    import sys, runpy
    sys.argv = ["push_artifacts_to_github.py", "--source", "/content", "--include-submissions"]
    try:
        runpy.run_path("/content/push_artifacts_to_github.py", run_name="__main__")
    except SystemExit as exc:
        print("code de sortie :", exc.code)

Colab — avec !python : un sous-processus n'a ni champ masqué ni Secrets, il faut
fournir le token autrement (secret exporté dans l'environnement, ou --token-file) :
    !python scripts/push_artifacts_to_github.py --source /content

Colab, en incluant le dossier de soumission :
    !python scripts/push_artifacts_to_github.py --source /content --include-submissions

Supprimer au passage un dossier de soumission obsolète (ancien slug) :
    python scripts/push_artifacts_to_github.py --source /content --include-submissions \\
        --prune-submissions

Local :
    python scripts/push_artifacts_to_github.py --repo . --source .

Vérifier sans rien publier :
    python scripts/push_artifacts_to_github.py --source . --no-push

Créer explicitement une branche inexistante :
    python scripts/push_artifacts_to_github.py --source . --branch nouvelle-branche --create-branch

Publier sur une autre branche / un autre dépôt :
    python scripts/push_artifacts_to_github.py --source . --branch main \
        --repo-url https://github.com/<user>/<repo>.git
"""

from __future__ import annotations

import argparse
import getpass
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/maick-code/tokenizer.git"
DEFAULT_BRANCH = "arena/01a092c6-tokenizer"   # branche de travail (main reste intacte)
DEFAULT_MESSAGE = "Artifacts: tokenizer.json + reports (run Colab)"
ARTIFACT_DIRS = ("models", "reports")
EXCLUDE_DIR_NAMES = {"__pycache__", ".ipynb_checkpoints", ".git"}
EXCLUDE_SUFFIXES = (".pyc", ".pyo", ".zip", ".tmp", ".log")

EXIT_OK, EXIT_ERROR, EXIT_MISSING, EXIT_UNSAFE = 0, 1, 2, 3


# --------------------------------------------------------------------------- #
# Utilitaires
# --------------------------------------------------------------------------- #
def log(message: str = "") -> None:
    print(message, flush=True)


def die(message: str, code: int) -> "NoReturn":  # noqa: F821
    log(f"\nERREUR : {message}")
    raise SystemExit(code)


def redact(text: str, token: str | None) -> str:
    """Supprime toute trace du token d'une sortie."""
    if not text:
        return ""
    if token:
        text = text.replace(token, "***")
    return text


def clone_dir_default() -> Path:
    if os.path.isdir("/content"):          # Google Colab
        return Path("/content/tokenizer")
    return Path.cwd() / ".push_clone"


# --------------------------------------------------------------------------- #
# Token
# --------------------------------------------------------------------------- #
def token_from_colab_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get("GITHUB_TOKEN")
        return value.strip() if value else None
    except Exception:
        return None


_COLAB_MASKED_FIELD_JS = r"""
new Promise((resolve) => {
  const box = document.createElement('div');
  box.style.cssText = 'font-family:monospace;padding:10px;margin-top:6px;'
                    + 'border:1px solid #c8c8c8;border-radius:6px;display:inline-block';
  const label = document.createElement('span');
  label.textContent = 'Colle ton token GitHub puis valide : ';
  const input = document.createElement('input');
  input.type = 'password';
  input.style.cssText = 'font-size:14px;padding:3px 5px;width:330px';
  const button = document.createElement('button');
  button.textContent = 'Enregistrer';
  button.style.cssText = 'margin-left:8px;padding:3px 12px';
  const done = () => {
    input.disabled = true; button.disabled = true;
    const value = input.value; box.remove(); resolve(value);
  };
  button.addEventListener('click', done);
  input.addEventListener('keydown', (event) => { if (event.key === 'Enter') done(); });
  box.appendChild(label); box.appendChild(input); box.appendChild(button);
  document.body.appendChild(box);
  input.focus();
})
"""


def token_from_colab_masked_field() -> str | None:
    """Champ de saisie masqué natif Colab (nécessite d'exécuter le script EN PROCESSUS).

    Fonctionne quand le script est lancé dans une cellule Python (``runpy``), pas
    avec ``!python`` : un sous-processus n'a pas accès à l'interface du notebook.
    """
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        value = output.eval_js(_COLAB_MASKED_FIELD_JS)
    except Exception as exc:
        log(f"Champ masqué Colab indisponible ({type(exc).__name__}) : repli sur la saisie classique.")
        return None
    if isinstance(value, str) and value.strip():
        log("Token saisi dans le champ masqué Colab (non affiché).")
        return value.strip().strip('"').strip("'")
    return None


def read_token(args: argparse.Namespace) -> str | None:
    """Token par ordre de priorité : --token-file, env, secret Colab, champ masqué Colab, saisie."""
    if args.token_file:
        path = Path(args.token_file)
        if not path.is_file():
            die(f"fichier de token introuvable : {path}", EXIT_ERROR)
        token = path.read_text(encoding="utf-8").strip()
        if token:
            log("Token lu depuis le fichier indiqué (--token-file).")
            return token

    for var in ("GITHUB_TOKEN", "GH_TOKEN"):
        token = os.environ.get(var)
        if token:
            log(f"Token récupéré depuis la variable d'environnement {var}.")
            return token.strip()

    token = token_from_colab_secret()
    if token:
        log("Token récupéré depuis le secret Colab 'GITHUB_TOKEN'.")
        return token

    if args.no_input:
        return None

    token = token_from_colab_masked_field()
    if token:
        return token

    prompt = "Colle ton token GitHub puis Entrée : "
    try:
        token = getpass.getpass(prompt)          # saisie masquée
    except Exception:
        try:
            token = input(prompt)                # repli si getpass indisponible
        except Exception:
            return None
    token = (token or "").strip().strip('"').strip("'")
    if token:
        log(f"Token saisi ({len(token)} caractères, non affiché).")
    return token or None


def authed_url(url: str, token: str | None) -> str:
    """URL https porteuse du token, uniquement pour github.com."""
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@")
    return url


# --------------------------------------------------------------------------- #
# Git
# --------------------------------------------------------------------------- #
def git(repo: Path | str | None, *args: str) -> subprocess.CompletedProcess:
    command = ["git"]
    if repo is not None:
        command += ["-C", str(repo)]
    return subprocess.run(command + list(args), capture_output=True, text=True)


def git_or_die(repo: Path | str | None, token: str | None, *args: str,
               what: str = "commande git") -> subprocess.CompletedProcess:
    result = git(repo, *args)
    if result.returncode != 0:
        die(f"{what} a échoué :\n{redact(result.stderr or result.stdout, token).strip()}",
            EXIT_ERROR)
    return result


# --------------------------------------------------------------------------- #
# Artefacts
# --------------------------------------------------------------------------- #
def best_model_tokenizer(source: Path) -> Path | None:
    """Chemin du tokenizer de la meilleure configuration du dernier balayage."""
    import json

    sweep = source / "reports" / "optimization_sweep.json"
    if not sweep.is_file():
        return None
    try:
        name = json.loads(sweep.read_text(encoding="utf-8"))["best"]["name"]
    except Exception:
        return None
    candidate = source / "models" / f"optimized_{name}" / "tokenizer.json"
    return candidate if candidate.is_file() else None


def prune_obsolete_submissions(source: Path) -> list[str]:
    """Supprime les dossiers submissions/<slug>/ obsolètes (tokenizer != meilleur modèle).

    Cas typique : après avoir renommé le SLUG, l'ancien dossier reste sur le disque et serait
    publié avec le nouveau — or le checker officiel exige exactement un répertoire de
    soumission. Seuls des dossiers dont le tokenizer.json diffère du meilleur modèle sont
    supprimés, et seulement s'il en reste plusieurs : le dossier courant est toujours conservé.
    """
    import hashlib

    subs = source / "submissions"
    if not subs.is_dir():
        return []
    dirs = sorted(p for p in subs.iterdir() if p.is_dir() and (p / "tokenizer.json").is_file())
    if len(dirs) < 2:
        return []
    best = best_model_tokenizer(source)
    digest = (lambda p: hashlib.sha256(p.read_bytes()).hexdigest())
    if best is not None and any(digest(d / "tokenizer.json") == digest(best) for d in dirs):
        keep = [d for d in dirs if digest(d / "tokenizer.json") == digest(best)]
    else:
        keep = [max(dirs, key=lambda d: d.stat().st_mtime)]
    removed = []
    for d in dirs:
        if d not in keep:
            shutil.rmtree(d)
            removed.append(d.name)
    if removed:
        log(f"dossiers de soumission obsolètes supprimés : {', '.join(removed)}")
        log(f"dossier conservé : {keep[0].name}")
    return removed


def collect_artifacts(source: Path, include_submissions: bool) -> list[str]:
    """Chemins relatifs (posix) des fichiers à publier, triés."""
    roots = list(ARTIFACT_DIRS) + (["submissions"] if include_submissions else [])
    files: list[str] = []
    for root in roots:
        base = source / root
        if not base.is_dir():
            continue
        for path in sorted(base.rglob("*")):
            if not path.is_file():
                continue
            parts = set(path.relative_to(source).parts)
            if parts & EXCLUDE_DIR_NAMES or path.name.startswith("."):
                continue
            if path.suffix.lower() in EXCLUDE_SUFFIXES:
                continue
            files.append(path.relative_to(source).as_posix())
    return files


def safety_checks(source: Path, files: list[str], force: bool) -> list[str]:
    """Contrôles avant publication. Retourne la liste des avertissements bloquants."""
    import json

    problems: list[str] = []

    baseline = source / "reports" / "baseline_bpe_10k.json"
    if baseline.is_file():
        try:
            status = json.loads(baseline.read_text(encoding="utf-8")).get("status")
            if status != "computed_on_official_dataset":
                problems.append(
                    f"reports/baseline_bpe_10k.json : status = {status!r} "
                    "(run non conforme au dataset officiel)")
        except Exception as exc:
            problems.append(f"reports/baseline_bpe_10k.json illisible : {exc}")

    sweep = source / "reports" / "optimization_sweep.json"
    if sweep.is_file():
        try:
            payload = json.loads(sweep.read_text(encoding="utf-8"))
            rows = (payload.get("dataset") or {}).get("validation_rows")
            if rows != 24_000:
                problems.append(
                    f"reports/optimization_sweep.json : validation_rows = {rows} "
                    "(attendu 24 000 : le balayage n'a pas tourné sur le vrai dataset)")
        except Exception as exc:
            problems.append(f"reports/optimization_sweep.json illisible : {exc}")

    if not any(f.startswith("models/") and f.endswith("tokenizer.json") for f in files):
        problems.append("aucun models/**/tokenizer.json trouvé dans les artefacts")

    # Une soumission = UN dossier. Un dossier obsolète laissé par une exécution antérieure
    # (ancien slug, par exemple après avoir renommé SLUG) rendrait la PR invalide : le
    # checker officiel exige exactement un répertoire `submissions/<slug>/`.
    subs = source / "submissions"
    if subs.is_dir():
        slugs = sorted(p.name for p in subs.iterdir()
                       if p.is_dir() and (p / "tokenizer.json").is_file())
        if len(slugs) > 1:
            problems.append(
                f"plusieurs dossiers de soumission dans submissions/ : {', '.join(slugs)} "
                "(un seul slug est autorisé par PR ; supprimez les dossiers obsolètes)")

    if problems and not force:
        log("\n" + "!" * 74)
        log("PUBLICATION REFUSÉE — les artefacts semblent ne pas venir d'un run réel :")
        for problem in problems:
            log(f"  - {problem}")
        log("Corrigez le run, ou relancez avec --force pour publier quand même.")
        log("!" * 74)
        raise SystemExit(EXIT_UNSAFE)

    if problems:
        log("\nAVERTISSEMENT (--force) :")
        for problem in problems:
            log(f"  - {problem}")
    return problems


# --------------------------------------------------------------------------- #
# Programme principal
# --------------------------------------------------------------------------- #
def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Publie models/ et reports/ vers votre dépôt GitHub (méthode token).",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("--source", default="/content" if os.path.isdir("/content") else ".",
                        help="répertoire contenant models/ et reports/ (défaut : /content ou .)")
    parser.add_argument("--repo", default=None,
                        help="clone git existant du dépôt (sinon clonage automatique)")
    parser.add_argument("--repo-url", default=DEFAULT_REPO_URL, help="URL https du dépôt")
    parser.add_argument("--branch", default=DEFAULT_BRANCH, help="branche cible")
    parser.add_argument("--message", default=DEFAULT_MESSAGE, help="message de commit")
    parser.add_argument("--token-file", default=None,
                        help="lire le token depuis un fichier (évite la saisie)")
    parser.add_argument("--no-input", action="store_true",
                        help="ne jamais demander le token de façon interactive")
    parser.add_argument("--no-push", action="store_true",
                        help="copier et commiter sans pousser")
    parser.add_argument("--prune-submissions", action="store_true",
                        help="supprimer les dossiers de soumission obsolètes (ancien slug) "
                             "avant publication : un seul slug est autorisé par PR")
    parser.add_argument("--include-submissions", action="store_true",
                        help="publier aussi submissions/**")
    parser.add_argument("--zip", action="store_true",
                        help="créer en plus une archive de secours dans --source")
    parser.add_argument("--force", action="store_true",
                        help="publier malgré les avertissements de conformité")
    parser.add_argument("--create-branch", action="store_true",
                        help="autoriser la création de la branche si elle n'existe pas")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    source = Path(args.source).resolve()
    branch = args.branch
    repo_url = args.repo_url

    log("=" * 74)
    log("Publication des artefacts vers GitHub")
    log("=" * 74)
    log(f"Source      : {source}")
    log(f"Dépôt       : {repo_url}")
    log(f"Branche     : {branch}")
    log(f"Artefacts   : {', '.join(ARTIFACT_DIRS + (('submissions',) if args.include_submissions else ()))}")
    log()

    if args.prune_submissions:
        prune_obsolete_submissions(source)

    files = collect_artifacts(source, args.include_submissions)
    if not files:
        die(f"aucun artefact trouvé dans {source} (attendu : models/, reports/)", EXIT_MISSING)

    log(f"{len(files)} fichier(s) à publier :")
    total = 0
    for rel in files:
        size = (source / rel).stat().st_size
        total += size
        log(f"  {size:>12,} o  {rel}")
    log(f"  {'-' * 12}")
    log(f"  {total:>12,} o  total")

    safety_checks(source, files, args.force)

    if args.zip:
        archive = source / "artifacts_backup.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
            for rel in files:
                handle.write(source / rel, rel)
        log(f"\nArchive de secours : {archive} ({archive.stat().st_size:,} o)")

    token = read_token(args)

    repo = Path(args.repo).resolve() if args.repo else clone_dir_default()

    if not (repo / ".git").exists():
        if not token:
            die("aucun token fourni et pas de clone local : impossible de cloner.", EXIT_ERROR)
        log(f"\nClone de {repo_url} (branche {branch}) dans {repo} ...")
        clone = subprocess.run(
            ["git", "clone", "--branch", branch, authed_url(repo_url, token), str(repo)],
            capture_output=True, text=True)
        if clone.returncode != 0:
            die("clonage impossible (token invalide, branche inexistante ou réseau) :\n"
                f"{redact(clone.stderr or clone.stdout, token).strip()}", EXIT_ERROR)
        log("clone : OK")
    else:
        log(f"\nClone existant réutilisé : {repo}")

    if token:
        check = git(repo, "ls-remote", "--heads", authed_url(repo_url, token), branch)
        if check.returncode != 0:
            die("authentification refusée : vérifiez la portée `repo` du token,"
                " sa date d'expiration et le nom de la branche.", EXIT_ERROR)
        log("authentification : OK")

    # La branche cible doit exister : sans ce contrôle, une faute de frappe
    # créerait silencieusement une nouvelle branche distante.
    exists = git(None, "ls-remote", "--heads",
                 authed_url(repo_url, token) if token else repo_url, branch)
    if exists.returncode == 0 and not exists.stdout.strip():
        if args.create_branch:
            log(f"branche '{branch}' absente du dépôt : elle sera créée (--create-branch).")
        else:
            die(f"la branche '{branch}' n'existe pas sur {repo_url}.\n"
                "Vérifiez le nom (--branch), ou utilisez --create-branch pour la créer.",
                EXIT_ERROR)
    elif exists.returncode != 0 and not token:
        log("(impossible de vérifier la branche sans token : le push tranchera.)")

    # --- resynchronisation ---------------------------------------------------
    # Un clone Colab réutilisé (ou un clone créé dans une session précédente) peut être
    # en retard sur la branche distante : le commit local ne serait alors pas un
    # fast-forward et le push serait refusé. On se replace d'abord sur la tête distante ;
    # les artefacts étant recopiés juste après, rien n'est perdu.
    fetch = git(repo, "fetch", authed_url(repo_url, token) if token else repo_url, branch)
    if fetch.returncode == 0:
        ancestor = git(repo, "merge-base", "--is-ancestor", "FETCH_HEAD", "HEAD")
        if ancestor.returncode == 0:
            log("clone à jour avec la branche distante.")
        else:
            local = git(repo, "rev-parse", "--short", "HEAD").stdout.strip()
            remote = git(repo, "rev-parse", "--short", "FETCH_HEAD").stdout.strip()
            log(f"clone en retard ({local}) sur la branche distante ({remote}) : "
                "resynchronisation sur la tête distante (les artefacts sont recopiés ensuite).")
            git_or_die(repo, token, "checkout", "-B", branch, "FETCH_HEAD",
                       what=f"git checkout -B {branch} {remote}")
    else:
        log("fetch impossible (réseau ?) : on tente le push tel quel.")

    for rel in files:
        destination = repo / rel
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / rel, destination)
    log(f"{len(files)} fichier(s) copié(s) dans le clone.")

    git(repo, "config", "user.name", "Artifact Publisher")
    git(repo, "config", "user.email", "publisher@users.noreply.github.com")
    for root in {Path(rel).parts[0] for rel in files}:
        git_or_die(repo, token, "add", root, what=f"git add {root}")

    commit = git(repo, "commit", "-m", args.message)
    if commit.returncode == 0:
        log("commit : OK")
    elif "nothing to commit" in (commit.stdout + commit.stderr):
        log("commit : rien de nouveau (artefacts identiques)")
    else:
        die(f"commit impossible :\n{redact(commit.stderr or commit.stdout, token).strip()}",
            EXIT_ERROR)

    if args.no_push:
        log("\n--no-push : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    if not token:
        log("\nAucun token : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    push = git(repo, "push", authed_url(repo_url, token), f"HEAD:{branch}")
    if push.returncode != 0:
        die(f"push refusé :\n{redact(push.stderr or push.stdout, token).strip()}", EXIT_ERROR)

    log("push : OK")
    log()
    log(f"Publié sur {repo_url} (branche {branch}).")
    if "github.com" in repo_url:
        slug = repo_url.rstrip("/").removesuffix(".git")
        log(f"Vérifiez : {slug}/tree/{branch}")
    return EXIT_OK


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# =============================================================================
# Exécution du script de publication (champ masqué Colab actif)
# =============================================================================
import runpy
import sys
from pathlib import Path

# Cellule autonome : elle fonctionne même si elle est la SEULE exécutée après un
# redémarrage de la VM (OUTPUT_ROOT, défini plus haut, n'existe alors pas encore).
SOURCE = Path(globals().get("OUTPUT_ROOT") or Path.cwd())
if not (SOURCE / "reports").is_dir():
    for candidate in (Path.cwd(), Path("/content")):
        if (candidate / "reports").is_dir():
            SOURCE = candidate
            break

SCRIPT_PATH = None
for candidate in (Path.cwd() / "push_artifacts_to_github.py",
                  SOURCE / "push_artifacts_to_github.py",
                  Path("/content/push_artifacts_to_github.py")):
    if candidate.is_file():
        SCRIPT_PATH = candidate.resolve()
        break
assert SCRIPT_PATH is not None, "la cellule %%writefile ci-dessus doit être exécutée d'abord"

# --prune-submissions : supprime un éventuel dossier de soumission obsolète (ancien slug),
# car le checker officiel exige exactement un répertoire submissions/<slug>/.
sys.argv = [
    "push_artifacts_to_github.py",
    "--source", str(SOURCE),
    "--branch", "arena/01a092c6-tokenizer",
    "--prune-submissions",
]

print("Exécution :", SCRIPT_PATH)
print("Arguments :", " ".join(sys.argv[1:]))
print("Un champ masqué « Colle ton token GitHub puis valide » va s'afficher.")
print()
try:
    runpy.run_path(str(SCRIPT_PATH), run_name="__main__")
except SystemExit as exc:
    print()
    print("Code de sortie du script :", exc.code,
          "| 0 = OK, 1 = erreur, 2 = artefacts manquants, 3 = publication refusée")
